# SigAlg's `RandomVariable` and `RandomVector` classes

In [ ]:
# If running in Google Colab, uncomment the line below and run this cell first.
# Also, for Mac+Chrome users, beware of a known bug with LaTeX redering in Colab: https://github.com/googlecolab/colabtools/issues/3192

# !pip install sigalg

The `RandomVariable` and `RandomVector` classes in SigAlg are the fundamental classes for representing random variables and random vectors. The API references are [here](https://johnmyers-phd.com/sigalg/api/modules/core/#sigalg.core.RandomVariable) and [here](https://johnmyers-phd.com/sigalg/api/modules/core/#sigalg.core.RandomVector).

## Mathematical definition

Given a probability space $(\Omega, \mathcal{F}, P)$, a *random vector* is an $\mathcal{F}$-measurable function $X: \Omega \to \mathbb{R}^d$, where $d$ is the *dimension* of the vector and $\mathbb{R}^d$ is equipped with its Borel $\sigma$-algebra. When $d=1$, the random vector reduces to a *random variable* $X: \Omega \to \mathbb{R}$.

The image $X(\omega) \in \mathbb{R}^d$ of a sample point $\omega \in \Omega$ is called a *feature vector* (or simply a *feature* when $d=1$). The function $X$ is said to be *$\mathcal{F}$-measurable* if for every Borel set $B \subset \mathbb{R}^d$, the preimage $X^{-1}(B) = \{\omega \in \Omega : X(\omega) \in B\}$ is an element of $\mathcal{F}$. If $\Omega$ is finite (as it always is, in SigAlg), so that $\mathcal{F}$ is determined by its atoms, then $X$ is $\mathcal{F}$-measurable if and only if $X$ is constant on the atoms of $\mathcal{F}$.

In SigAlg, an instance `X` of `RandomVector` represents such a random vector. The instance carries:
- A `domain` attribute representing $\Omega$
- A `data` attribute representing the mapping $\omega \mapsto X(\omega)$
- A `probability_measure` attribute representing $P$
- An `index` attribute labeling the components of the vector

## API examples

### Creating random vectors and variables

#### From dictionaries

Begin by defining a sample space $\Omega = \{0,1,2,3\}$.

In [ ]:
from sigalg.core import RandomVariable, RandomVector, SampleSpace

Omega = SampleSpace().from_sequence(size=4)

print(Omega)

Sample space 'Omega':
[0, 1, 2, 3]


Create a 2-dimensional random vector from a dictionary:

In [3]:
X = RandomVector(domain=Omega, name="X").from_dict(
    {
        0: (1, 2),
        1: (3, 4),
        2: (5, 6),
        3: (7, 8),
    }
)
print(X)

Random vector 'X':
feature  X_0  X_1
sample           
0          1    2
1          3    4
2          5    6
3          7    8


Create a random variable (1-dimensional) from a dictionary:

In [4]:
Y = RandomVariable(domain=Omega, name="Y").from_dict(
    {
        0: 1,
        1: -2,
        2: 3,
        3: -1,
    }
)
print(Y)

Random variable 'Y':
        Y
sample   
0       1
1      -2
2       3
3      -1


#### From `pd.DataFrame` and `pd.Series` objects

Create a random vector from a data frame:

In [ ]:
import pandas as pd

df = pd.DataFrame([[1, 2], [3, 4], [5, 6]], index=["a", "b", "c"], columns=["A", "B"])

Z = RandomVector(name="Z").from_pandas(df)
print(Z)

Random vector 'Z':
   A  B
a  1  2
b  3  4
c  5  6


Create a random variable from a series:

In [6]:
s = pd.Series([10, 20, 30, 40], index=["a", "b", "c", "d"])

W = RandomVariable(name="W").from_pandas(s)
print(W)

Random variable 'W':
    W
a  10
b  20
c  30
d  40


#### From `np.ndarray` objects

Create a random vector from an array:

In [7]:
import numpy as np

arr = np.array([[1, 2, 3], [4, 5, 6]])

U = RandomVector(name="U").from_numpy(arr)
print(U)

Random vector 'U':
   0  1  2
0  1  2  3
1  4  5  6


#### From random sampling

Generate a random vector with integer components uniformly sampled from a range:

In [8]:
rng = np.random.default_rng(42)

A = RandomVector(domain=Omega, name="A").from_randint(
    low=0, high=10, dim=3, random_state=rng
)
print(A)

Random vector 'A':
        0  1  2
sample         
0       0  7  6
1       4  4  8
2       0  6  2
3       0  5  9


Generate a random vector with components sampled from a normal distribution:

In [9]:
B = RandomVector(domain=Omega, name="B").from_randnorm(
    loc=1.0, scale=2.0, dim=2, random_state=rng
)
print(B)

Random vector 'B':
               0         1
sample                    
0       1.255681  0.367515
1       0.966398 -0.706088
2       2.758796  2.555584
3       1.132061  3.254482


#### From constant values

Create a constant random variable:

In [10]:
C = RandomVariable(domain=Omega, name="C").from_constant(5)
print(C)

Random variable 'C':
        C
sample   
0       5
1       5
2       5
3       5


#### Indicator random variables

Create an indicator random variable for an event:

In [11]:
E = Omega.get_event([0, 1], name="E")
I = RandomVariable.indicator_of(event=E)
print(I)

Random variable 'I_E':
        I_E
sample     
0         1
1         1
2         0
3         0


### Properties of random vectors

#### Domain, dimension, and data

Access the domain (sample space) of a random vector:

In [12]:
print(f"Domain of X: {X.domain}\n")
print(f"Dimension of X: {X.dimension}\n")
print(f"Data of X:\n{X.data}")

Domain of X: Sample space 'Omega':
[0, 1, 2, 3]

Dimension of X: 2

Data of X:
feature  X_0  X_1
sample           
0          1    2
1          3    4
2          5    6
3          7    8


#### Index and naming

Access and modify the index (component labels) and name:

In [13]:
print(f"Index of X: {list(X.index)}\n")
print(f"Name of X: {X.name}\n")

X_renamed = X.with_name("X_new", modify_index=True)
print(X_renamed)

Index of X: ['X_0', 'X_1']

Name of X: X

Random vector 'X_new':
feature  X_new_0  X_new_1
sample                   
0              1        2
1              3        4
2              5        6
3              7        8


#### Probability measure

Set a probability measure on a random vector:

In [14]:
from sigalg.core import ProbabilityMeasure

P = ProbabilityMeasure(sample_space=Omega).from_dict(
    {
        0: 0.1,
        1: 0.2,
        2: 0.3,
        3: 0.4,
    }
)

X.prob_measure = P
print(X.prob_measure)

Probability measure 'P':
        probability
sample             
0               0.1
1               0.2
2               0.3
3               0.4


### Calling random vectors

#### Evaluating at a single point

Evaluate a random vector at a single sample point:

In [15]:
print(X(0), "\n")
print(Y(2))

Feature vector of '0':
         0
feature   
X_new_0  1
X_new_1  2 

3


#### Restricting to an event

Restrict a random vector to an event:

In [16]:
X.with_name("X", modify_index=True)
A = Omega.get_event([0, 2], name="A")
X_A = X(A)
print(X_A)

Random vector 'X|A':
feature  X_0  X_1
sample           
0          1    2
2          5    6


You can also pass a list directly:

In [17]:
X_restricted = X([1, 3])
print(X_restricted)

Random vector 'X|event':
feature  X_0  X_1
sample           
1          3    4
3          7    8


### Accessing components

#### Extracting individual components

Extract a single component as a random variable:

In [18]:
X_0 = X.get_component_rv("X_0")
print(X_0)

Random variable 'X_0':
        X_0
sample     
0         1
1         3
2         5
3         7


#### Extracting sub-vectors

Extract a sub-vector with selected components:

In [19]:
# Create a 3-dimensional random vector first
V = RandomVector(domain=Omega, name="V").from_dict(
    {
        0: (1, 2, 3),
        1: (4, 5, 6),
        2: (7, 8, 9),
        3: (10, 11, 12),
    }
)
V.prob_measure = P

V_sub = V.get_sub_vector(["V_0", "V_2"])
print(V_sub)

Random vector 'V_sub':
feature  V_0  V_2
sample           
0          1    3
1          4    6
2          7    9
3         10   12


#### Extracting constant values

For constant random vectors, extract the value using the `item` method:

In [20]:
const = RandomVariable(domain=Omega, name="const").from_constant(42)
print(f"Constant random variable:\n{const}\n")
print(f"Extracted value: {const.item()}")

Constant random variable:
Random variable 'const':
        const
sample       
0          42
1          42
2          42
3          42

Extracted value: 42


### Algebraic operations

#### Basic arithmetic

Random vectors support element-wise arithmetic operations:

In [21]:
X1 = RandomVector(domain=Omega, name="X1").from_dict(
    {
        0: (1, 2),
        1: (3, 4),
        2: (5, 6),
        3: (7, 8),
    }
)
X2 = RandomVector(domain=Omega, name="X2").from_dict(
    {
        0: (2, 1),
        1: (1, 2),
        2: (3, 2),
        3: (1, 1),
    }
)

print("Addition:")
print(X1 + X2, "\n")

print("Subtraction:")
print(X1 - X2, "\n")

print("Multiplication:")
print(X1 * X2, "\n")

print("Division:")
print(X1 / X2, "\n")

print("Power:")
print(X1 ** 2, "\n")

print("Linear combination:")
print(2 * X1 - 3 * X2)

Addition:


Random vector '(X1+X2)':
feature  (X1+X2)_0  (X1+X2)_1
sample                       
0                3          3
1                4          6
2                8          8
3                8          9 

Subtraction:
Random vector '(X1-X2)':
feature  (X1-X2)_0  (X1-X2)_1
sample                       
0               -1          1
1                2          2
2                2          4
3                6          7 

Multiplication:
Random vector '(X1*X2)':
feature  (X1*X2)_0  (X1*X2)_1
sample                       
0                2          2
1                3          8
2               15         12
3                7          8 

Division:
Random vector '(X1/X2)':
feature  (X1/X2)_0  (X1/X2)_1
sample                       
0         0.500000        2.0
1         3.000000        2.0
2         1.666667        3.0
3         7.000000        8.0 

Power:
Random vector '(X1**2)':
feature  (X1**2)_0  (X1**2)_1
sample                       
0                1          4
1          

#### NumPy functions

Apply NumPy universal functions to random vectors:

In [22]:
Z = RandomVector(domain=Omega, name="Z").from_dict(
    {
        0: (0, np.pi),
        1: (np.pi / 2, 3 * np.pi / 2),
        2: (np.pi, 2 * np.pi),
        3: (np.pi / 4, 5 * np.pi / 4),
    }
)

print(np.sin(Z).round(2), "\n")

Y = RandomVariable(domain=Omega, name="Y").from_dict(
    {
        0: 0,
        1: 1,
        2: 2,
        3: 3,
    }
)
print(np.exp(Y).round(2))

Random vector 'sin(Z)':
feature  sin(Z_0)  sin(Z_1)
sample                     
0            0.00      0.00
1            1.00     -1.00
2            0.00     -0.00
3            0.71     -0.71 

Random variable 'exp(Y)':
        exp(Y)
sample        
0         1.00
1         2.72
2         7.39
3        20.09


#### Custom functions

Apply custom functions that operate on feature vectors:

In [23]:
from sigalg.core import FeatureVector


def sum_plus_one(vec: FeatureVector) -> float:
    """Sum the entries of a feature vector and add 1."""
    return vec.sum() + 1

result = X.apply_to_features(sum_plus_one)
print(result)

Random variable 'X_apply':
        X_apply
sample         
0             4
1             8
2            12
3            16


### Comparisons

Compare random vectors element-wise:

In [24]:
print("X1 < X2:")
print(X1 < X2, "\n")

print("X1 <= X2:")
print(X1 <= X2, "\n")

print("X1 > X2:")
print(X1 > X2, "\n")

print("X1 >= 5:")
print(X1 >= 5, "\n")

X1 < X2:
Random vector '(X1 < X2)':
feature  (X1 < X2)_0  (X1 < X2)_1
sample                           
0               True        False
1              False        False
2              False        False
3              False        False 

X1 <= X2:
Random vector '(X1 <= X2)':
feature  (X1 <= X2)_0  (X1 <= X2)_1
sample                             
0                True         False
1               False         False
2               False         False
3               False         False 

X1 > X2:
Random vector '(X1 > X2)':
feature  (X1 > X2)_0  (X1 > X2)_1
sample                           
0              False         True
1               True         True
2               True         True
3               True         True 

X1 >= 5:
Random vector '(X1 >= 5)':
feature  (X1 >= 5)_0  (X1 >= 5)_1
sample                           
0              False        False
1              False        False
2               True         True
3               True         True 



Check if any or all comparisons are true:

In [25]:
print(f"Are any entries of X1 less than X2? {(X1 < X2).any()}\n")
print(f"Are all entries of X1 less than 10? {(X1 < 10).all()}")

Are any entries of X1 less than X2? True

Are all entries of X1 less than 10? True


### Measurability

Check if a random vector is measurable with respect to a $\sigma$-algebra:

In [26]:
from sigalg.core import SigmaAlgebra

# Create two random vectors
M = RandomVector(domain=Omega, name="M").from_dict(
    {
        0: (1, 2),
        1: (3, 4),
        2: (3, 4),
        3: (3, 4),
    }
)

N = RandomVector(domain=Omega, name="N").from_dict(
    {
        0: (1, 2),
        1: (3, 4),
        2: (5, 6),
        3: (7, 8),
    }
)

# Create a sigma-algebra with atoms {0}, {1,2,3}
F = SigmaAlgebra(sample_space=Omega, name="F").from_dict(
    {
        0: 0,
        1: 1,
        2: 1,
        3: 1,
    }
)

print(f"Is M F-measurable? {M.is_measurable(F)}")
print(f"Is N F-measurable? {N.is_measurable(F)}")

Is M F-measurable? True
Is N F-measurable? False


The random vector `M` is $\mathcal{F}$-measurable because it is constant on the atom $\{1, 2, 3\}$, while `N` is not measurable because it takes different values on this atom.